# Mitigating VQE Errors on H₂ with EMRG

EMRG analyzes a quantum circuit and generates ready-to-run Mitiq error mitigation code, selecting between ZNE, PEC, and CDR based on circuit characteristics. This notebook builds a 4-qubit H₂ VQE ansatz, runs it through EMRG's three-technique pipeline, and shows how each technique applies to the same circuit.

In [ ]:
# pip install emrg

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

from emrg import analyze_circuit, generate_recipe

## Build the H₂ VQE ansatz

Hardware-efficient ansatz: 4 qubits, 2 layers of RY rotations and CNOT entanglement in the STO-3G basis. Parameters are bound to fixed values for demonstration. This ansatz structure follows patterns identified in autonomous VQE architecture search ([github.com/FedorShind/autoresearch-qc](https://github.com/FedorShind/autoresearch-qc)).

In [ ]:
rng = np.random.default_rng(42)
n_qubits = 4
n_layers = 2

qc = QuantumCircuit(n_qubits, n_qubits)
for _ in range(n_layers):
    for q in range(n_qubits):
        qc.ry(float(rng.uniform(0, 2 * np.pi)), q)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
qc.measure(range(n_qubits), range(n_qubits))

print(f"Qubits: {qc.num_qubits}")
print(f"Depth:  {qc.depth()}")
print(qc.draw())

## Analyze the circuit

`analyze_circuit` extracts features from the circuit using static analysis only — no simulation.

In [ ]:
features = analyze_circuit(qc)

print(f"Depth:               {features.depth}")
print(f"Total gates:         {features.total_gate_count}")
print(f"Multi-qubit gates:   {features.multi_qubit_gate_count}")
print(f"Noise estimate:      {features.estimated_noise_factor}")
print(f"Noise category:      {features.noise_category}")
print(f"Non-Clifford count:  {features.non_clifford_count}")
print(f"Non-Clifford frac:   {features.non_clifford_fraction:.4f}")
print(f"PEC overhead est:    {features.pec_overhead_estimate:.2f}")
print(f"Layer heterogeneity: {features.layer_heterogeneity:.4f}")

The RY gates with arbitrary angles are non-Clifford (57% of total gates). The depth of 8 falls below CDR's minimum depth threshold of 10, so EMRG will route this circuit to ZNE rather than CDR. The PEC overhead of ~2.3 is low enough for PEC if a noise model is available.

## Generate the default recipe

`generate_recipe` chains analysis, technique selection, and code generation in a single call.

In [ ]:
result = generate_recipe(qc, explain=True)

print(f"Technique:     {result.recipe.technique}")
print(f"Factory:       {result.recipe.factory_name}")
print(f"Scale factors: {result.recipe.scale_factors}")
print(f"Scaling:       {result.recipe.scaling_method}")
print()
print(result.code)

EMRG selected ZNE with LinearFactory because the depth (8) is below 20 and the multi-qubit gate count (6) is low. This is the most conservative ZNE configuration — three scale factors, linear extrapolation, global folding.

## PEC recipe

PEC (Probabilistic Error Cancellation) provides unbiased error correction at the cost of exponential sampling overhead. It requires a noise model. Setting `noise_model_available=True` enables PEC consideration.

In [ ]:
pec_result = generate_recipe(qc, noise_model_available=True, explain=True)

print(f"Technique: {pec_result.recipe.technique}")
print(f"Overhead:  {pec_result.features.pec_overhead_estimate:.2f}x")
print()
for line in pec_result.rationale:
    print(f"  - {line}")

EMRG selected PEC here because the circuit is shallow (depth 8, below the PEC threshold of 30) and the overhead (~2.3x) is well below the 1000x cutoff. PEC has higher priority than CDR or ZNE when its conditions are met.

## CDR recipe

CDR (Clifford Data Regression) replaces non-Clifford gates with Clifford substitutes to create training circuits that can be simulated classically. A regression model fitted on these training results corrects the noisy output. CDR sits between ZNE (approximate, no noise model needed) and PEC (exact, expensive) in terms of accuracy and cost.

In [ ]:
cdr_result = generate_recipe(qc, technique="cdr")

kwargs = cdr_result.recipe.factory_kwargs
print(f"Technique:          {cdr_result.recipe.technique}")
print(f"Training circuits:  {kwargs.get('num_training_circuits')}")
print(f"Fit method:         {kwargs.get('fit_method')}")
print()
print(cdr_result.code)

CDR was forced here with `technique="cdr"`. EMRG would not auto-select CDR for this circuit because the depth (8) is below CDR's minimum threshold of 10. For circuits with depth 10–40 and more than 20% non-Clifford gates, CDR is selected automatically.

## Preview mode

`preview=True` runs a noisy simulation (Cirq DensityMatrixSimulator with depolarizing noise) and compares ideal, noisy, and mitigated expectation values. Requires `pip install emrg[preview]`.

In [ ]:
preview_result = generate_recipe(qc, preview=True, noise_level=0.01)

if preview_result.preview and preview_result.preview.ideal_value is not None:
    p = preview_result.preview
    print(f"Technique:  {p.technique}")
    print(f"Ideal:      {p.ideal_value:+.4f}")
    print(f"Noisy:      {p.noisy_value:+.4f}  (error: {p.noisy_error:.4f})")
    print(f"Mitigated:  {p.mitigated_value:+.4f}  (error: {p.mitigated_error:.4f})")
    print(f"Reduction:  {p.error_reduction:.1f}x")
else:
    p = preview_result.preview
    warning = p.warning if p else "cirq not installed"
    print(f"Preview skipped: {warning}")

## Summary

EMRG analyzed a 4-qubit H₂ VQE circuit and selected ZNE (LinearFactory) as the default technique, PEC when a noise model is available, and CDR when forced. The three techniques cover different tradeoffs: ZNE is approximate but universal, PEC is unbiased but expensive, CDR is accurate on non-Clifford-heavy circuits but needs a classical simulator.

- [EMRG on GitHub](https://github.com/FedorShind/EMRG)
- [EMRG on PyPI](https://pypi.org/project/emrg/)
- [Mitiq documentation](https://mitiq.readthedocs.io/)